# SchemaGuard на уже готовых Spider-предсказаниях 

Этот нотутбук проверяет, улучшает ли SchemaGuard старые предсказания модели, которые уже давали хороший результат

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [22]:
%pip install -q -U unsloth unsloth_zoo
%pip install -q -U "sqlglot>=25.0.0" "sqlparse>=0.5.0" "nltk>=3.8.0" "tqdm>=4.66.0"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [23]:
import gc
import json
import os
import re
import shutil
import sqlite3
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import torch
from tqdm.auto import tqdm
from sqlglot import exp, parse_one
from sqlglot.errors import ParseError

from unsloth import FastLanguageModel

os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not torch.cuda.is_available():
    raise RuntimeError("Нужен GPU runtime. В Kaggle включите Accelerator -> GPU.")

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
GPU: Tesla T4
BF16 supported: False


In [24]:
RUN_NAME = "old_predictions_schema_guard"

OLD_PREDICTIONS_PATH = Path(
    "/kaggle/input/models/olllllllll/lora7b-150/pytorch/default/1/dev_predictions.jsonl"
)

EVAL_PROMPTS_FILE = Path(
    "/kaggle/input/datasets/olllllllll/spider/dev_eval_prompts.jsonl"
)

# Оригинальный Spide
SPIDER_DATA_DIR = Path(
    "/kaggle/input/datasets/olllllllll/spider-orig/spider"
)
SPIDER_DB_DIR = SPIDER_DATA_DIR / "database"
SPIDER_TABLES = SPIDER_DATA_DIR / "tables.json"

ADAPTER_DIR = Path(
    "/kaggle/input/models/olllllllll/lora7b-150/pytorch/default/1"
)

BASE_MODEL_OVERRIDE = None

MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 128
ALLOW_REPAIR_PROMPT_TRUNCATION = False

REPAIR_WARNINGS = False #True, если хотим отправлять вообще все подозрительные

OUTPUT_DIR = Path("/kaggle/working") / RUN_NAME
PRED_DIR = OUTPUT_DIR / "predictions"
EVAL_DIR = OUTPUT_DIR / "eval_files"
LOG_DIR = OUTPUT_DIR / "logs"

for directory in [OUTPUT_DIR, PRED_DIR, EVAL_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)

OUTPUT_DIR: /kaggle/working/old_predictions_schema_guard


In [25]:
def require_path(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Не найдено: {description}\n{path}")

for path, description in [
    (OLD_PREDICTIONS_PATH, "старые предсказания"),
    (EVAL_PROMPTS_FILE, "Spider dev_eval_prompts"),
    (SPIDER_DB_DIR, "Spider database"),
    (SPIDER_TABLES, "Spider tables.json"),
    (ADAPTER_DIR, "LoRA adapter directory"),
    (ADAPTER_DIR / "adapter_config.json", "adapter_config.json"),
]:
    require_path(path, description)

print("Все обязательные пути существуют.")

Все обязательные пути существуют.


In [26]:
PREDICTION_KEYS = [
    "pred_sql",
    "baseline_sql",
    "clean_pred_sql",
    "sql_prediction",
    "prediction",
    "sql",
]


def load_json_or_jsonl(path: Path):
    text = path.read_text(encoding="utf-8").strip()

    if not text:
        raise ValueError(f"Файл пуст: {path}")

    if text.startswith("["):
        return json.loads(text)

    return [
        json.loads(line)
        for line in text.splitlines()
        if line.strip()
    ]


def extract_prediction(row: dict, row_index: int) -> str:
    for key in PREDICTION_KEYS:
        if key in row:
            return str(row[key]).strip()

    raise ValueError(
        f"Строка {row_index}: не найдено поле предсказания. "
        f"Ожидалось одно из: {PREDICTION_KEYS}"
    )


def load_old_predictions(
    path: Path,
    eval_rows: list[dict],
) -> list[str]:
    suffix = path.suffix.lower()

    if suffix not in {".json", ".jsonl"}:
        predictions = path.read_text(
            encoding="utf-8"
        ).splitlines()

        if len(predictions) != len(eval_rows):
            raise ValueError(
                "Для .sql / .txt порядок строк должен совпадать с dev. "
                f"dev={len(eval_rows)}, predictions={len(predictions)}"
            )

        return predictions

    rows = load_json_or_jsonl(path)

    # Если в JSONL есть id, сопоставляем по id
    if rows and all("id" in row for row in rows):
        by_id = {
            str(row["id"]): extract_prediction(row, index)
            for index, row in enumerate(rows)
        }

        missing_ids = [
            str(item["id"])
            for item in eval_rows
            if str(item["id"]) not in by_id
        ]

        if missing_ids:
            raise ValueError(
                "В старых предсказаниях не хватает id: "
                + ", ".join(missing_ids[:20])
            )

        return [
            by_id[str(item["id"])]
            for item in eval_rows
        ]

    predictions = [
        extract_prediction(row, index)
        for index, row in enumerate(rows)
    ]

    if len(predictions) != len(eval_rows):
        raise ValueError(
            f"dev={len(eval_rows)}, predictions={len(predictions)}"
        )

    return predictions


eval_examples = load_json_or_jsonl(EVAL_PROMPTS_FILE)

required_fields = {"id", "db_id", "question", "sql", "text"}

for index, item in enumerate(eval_examples):
    missing = required_fields - set(item)

    if missing:
        raise ValueError(
            f"dev строка {index}: отсутствуют поля {sorted(missing)}"
        )

old_predictions = load_old_predictions(
    OLD_PREDICTIONS_PATH,
    eval_examples,
)

print("Spider dev examples:", len(eval_examples))
print("Old predictions:", len(old_predictions))
print("First prediction:", old_predictions[0])

Spider dev examples: 1034
Old predictions: 1034
First prediction: SELECT count(*) FROM singer


In [27]:
def load_spider_schemas(path: Path) -> dict:
    raw_schemas = json.loads(
        path.read_text(encoding="utf-8")
    )

    schemas = {}

    for db in raw_schemas:
        table_names = db["table_names_original"]
        column_names = db["column_names_original"]

        tables = {
            table_name.lower(): {
                "original_name": table_name,
                "columns": set(),
            }
            for table_name in table_names
        }

        column_index_to_ref = {}

        for column_index, (table_index, column_name) in enumerate(
            column_names
        ):
            if table_index == -1:
                continue

            table_name = table_names[table_index].lower()
            column_name = column_name.lower()

            tables[table_name]["columns"].add(column_name)
            column_index_to_ref[column_index] = (
                table_name,
                column_name,
            )

        foreign_keys = set()

        for left_index, right_index in db.get("foreign_keys", []):
            left_ref = column_index_to_ref.get(left_index)
            right_ref = column_index_to_ref.get(right_index)

            if left_ref and right_ref:
                foreign_keys.add(
                    frozenset([left_ref, right_ref])
                )

        schemas[db["db_id"]] = {
            "tables": tables,
            "foreign_keys": foreign_keys,
        }

    return schemas


SPIDER_SCHEMAS = load_spider_schemas(SPIDER_TABLES)

print("Loaded Spider schemas:", len(SPIDER_SCHEMAS))

Loaded Spider schemas: 166


In [28]:
@dataclass
class GuardReport:
    parseable: bool
    sqlite_valid: bool
    sqlite_error: str
    parse_error: str
    unknown_tables: list[str]
    unknown_aliases: list[str]
    unknown_columns: list[str]
    ambiguous_columns: list[str]
    suspicious_joins: list[str]

    @property
    def hard_error_count(self) -> int:
        return (
            int(not self.sqlite_valid)
            + len(self.unknown_tables)
            + len(self.unknown_aliases)
            + len(self.unknown_columns)
        )

    @property
    def warning_count(self) -> int:
        return (
            len(self.ambiguous_columns)
            + len(self.suspicious_joins)
        )

    @property
    def score(self) -> int:
        return (
            100 * self.hard_error_count
            + 5 * len(self.suspicious_joins)
            + len(self.ambiguous_columns)
        )


def get_db_path(db_id: str) -> Path:
    path = SPIDER_DB_DIR / db_id / f"{db_id}.sqlite"

    if not path.exists():
        raise FileNotFoundError(f"SQLite DB не найден: {path}")

    return path


def explain_sql(db_id: str, sql: str) -> tuple[bool, str]:
    if not sql.strip():
        return False, "empty SQL"

    try:
        with sqlite3.connect(str(get_db_path(db_id))) as connection:
            # В Spider встречаются строки с некорректной UTF-8-последовательностью.
            connection.text_factory = lambda raw: raw.decode(errors="ignore")
            connection.execute("EXPLAIN QUERY PLAN " + sql)

        return True, ""

    except Exception as error:
        return False, str(error)


def collect_aliases(
    tree: exp.Expression,
    schema_tables: dict,
) -> tuple[dict[str, str], list[str]]:
    alias_to_table = {}
    unknown_tables = []

    for table_node in tree.find_all(exp.Table):
        table_name = table_node.name.lower()
        alias_name = table_node.alias_or_name.lower()

        if table_name not in schema_tables:
            unknown_tables.append(table_name)
            continue

        alias_to_table[alias_name] = table_name

        if not table_node.alias:
            alias_to_table.setdefault(table_name, table_name)

    return alias_to_table, sorted(set(unknown_tables))


def resolve_column(
    column: exp.Column,
    alias_to_table: dict[str, str],
    schema_tables: dict,
) -> tuple[Optional[tuple[str, str]], Optional[str]]:
    column_name = column.name.lower()
    qualifier = column.table.lower() if column.table else ""

    if qualifier:
        if qualifier not in alias_to_table:
            return None, f"unknown alias: {qualifier}"

        table_name = alias_to_table[qualifier]

        if column_name not in schema_tables[table_name]["columns"]:
            return None, f"unknown column: {table_name}.{column_name}"

        return (table_name, column_name), None

    candidate_tables = sorted(
        {
            table_name
            for table_name in alias_to_table.values()
            if column_name in schema_tables[table_name]["columns"]
        }
    )

    if not candidate_tables:
        return None, f"unknown unqualified column: {column_name}"

    if len(candidate_tables) > 1:
        return (
            None,
            f"ambiguous unqualified column: {column_name}; "
            f"possible tables: {candidate_tables}",
        )

    return (candidate_tables[0], column_name), None

def is_probable_double_quoted_literal(
    column: exp.Column,
) -> bool:

    if column.table:
        return False

    identifier = column.this

    if not isinstance(identifier, exp.Identifier):
        return False

    if not identifier.args.get("quoted"):
        return False

    parent = column.parent

    if isinstance(
        parent,
        (
            exp.EQ,
            exp.NEQ,
            exp.GT,
            exp.GTE,
            exp.LT,
            exp.LTE,
            exp.Is,
        ),
    ):
        return parent.right is column

    if isinstance(parent, exp.Like):
        return parent.expression is column

    if isinstance(parent, exp.In):
        return True

    return False

def find_suspicious_joins(
    tree: exp.Expression,
    alias_to_table: dict[str, str],
    schema_tables: dict,
    foreign_keys: set,
) -> list[str]:
    issues = []

    for join_node in tree.find_all(exp.Join):
        on_expression = join_node.args.get("on")

        if on_expression is None:
            issues.append(
                f"JOIN without ON: {join_node.sql(dialect='sqlite')}"
            )
            continue

        for equality in on_expression.find_all(exp.EQ):
            if not isinstance(equality.left, exp.Column):
                continue

            if not isinstance(equality.right, exp.Column):
                continue

            left_ref, left_error = resolve_column(
                equality.left,
                alias_to_table,
                schema_tables,
            )

            right_ref, right_error = resolve_column(
                equality.right,
                alias_to_table,
                schema_tables,
            )

            if left_error or right_error:
                continue

            if left_ref[0] == right_ref[0]:
                continue

            edge = frozenset([left_ref, right_ref])

            if edge not in foreign_keys:
                issues.append(
                    f"{left_ref[0]}.{left_ref[1]} "
                    f"= {right_ref[0]}.{right_ref[1]}"
                )

    return sorted(set(issues))


def guard_sql(db_id: str, sql: str) -> GuardReport:
    sqlite_valid, sqlite_error = explain_sql(db_id, sql)

    schema = SPIDER_SCHEMAS[db_id]
    schema_tables = schema["tables"]

    try:
        tree = parse_one(sql, read="sqlite")

    except ParseError as error:
        return GuardReport(
            parseable=False,
            sqlite_valid=sqlite_valid,
            sqlite_error=sqlite_error,
            parse_error=str(error),
            unknown_tables=[],
            unknown_aliases=[],
            unknown_columns=[],
            ambiguous_columns=[],
            suspicious_joins=[],
        )

    alias_to_table, unknown_tables = collect_aliases(
        tree,
        schema_tables,
    )

    unknown_aliases = []
    unknown_columns = []
    ambiguous_columns = []

    for column in tree.find_all(exp.Column):
        if is_probable_double_quoted_literal(column):
            continue

        _, error = resolve_column(
            column,
            alias_to_table,
            schema_tables,
        )

        if not error:
            continue

        if error.startswith("unknown alias"):
            unknown_aliases.append(error)

        elif error.startswith("ambiguous"):
            ambiguous_columns.append(error)

        else:
            unknown_columns.append(error)

    suspicious_joins = find_suspicious_joins(
        tree=tree,
        alias_to_table=alias_to_table,
        schema_tables=schema_tables,
        foreign_keys=schema["foreign_keys"],
    )

    return GuardReport(
        parseable=True,
        sqlite_valid=sqlite_valid,
        sqlite_error=sqlite_error,
        parse_error="",
        unknown_tables=sorted(set(unknown_tables)),
        unknown_aliases=sorted(set(unknown_aliases)),
        unknown_columns=sorted(set(unknown_columns)),
        ambiguous_columns=sorted(set(ambiguous_columns)),
        suspicious_joins=suspicious_joins,
    )


def should_attempt_repair(report: GuardReport) -> bool:
    if report.hard_error_count > 0:
        return True

    return REPAIR_WARNINGS and report.warning_count > 0

In [29]:
audit_reports = [
    guard_sql(item["db_id"], sql)
    for item, sql in tqdm(
        zip(eval_examples, old_predictions),
        total=len(eval_examples),
    )
]

audit_summary = {
    "examples": len(audit_reports),
    "sqlite_invalid": sum(
        not report.sqlite_valid
        for report in audit_reports
    ),
    "unknown_tables": sum(
        len(report.unknown_tables)
        for report in audit_reports
    ),
    "unknown_aliases": sum(
        len(report.unknown_aliases)
        for report in audit_reports
    ),
    "unknown_columns": sum(
        len(report.unknown_columns)
        for report in audit_reports
    ),
    "ambiguous_columns": sum(
        len(report.ambiguous_columns)
        for report in audit_reports
    ),
    "suspicious_joins": sum(
        len(report.suspicious_joins)
        for report in audit_reports
    ),
    "repair_candidates": sum(
        should_attempt_repair(report)
        for report in audit_reports
    ),
}

print(json.dumps(
    audit_summary,
    ensure_ascii=False,
    indent=2,
))

  0%|          | 0/1034 [00:00<?, ?it/s]

{
  "examples": 1034,
  "sqlite_invalid": 51,
  "unknown_tables": 10,
  "unknown_aliases": 10,
  "unknown_columns": 45,
  "ambiguous_columns": 59,
  "suspicious_joins": 50,
  "repair_candidates": 52
}


In [30]:
from peft import PeftModel

adapter_config = json.loads(
    (ADAPTER_DIR / "adapter_config.json").read_text(
        encoding="utf-8"
    )
)

base_model_name = (
    BASE_MODEL_OVERRIDE
    or adapter_config.get("base_model_name_or_path")
)

if not base_model_name:
    raise ValueError(
        "Не удалось определить базовую модель"
        "Укажите модель вручную"
    )

print("Adapter directory:", ADAPTER_DIR)
print("Base model:", base_model_name)

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(ADAPTER_DIR),
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    print("Loaded adapter directly through FastLanguageModel")

except Exception as direct_load_error:
    print("Direct adapter load failed:")
    print(type(direct_load_error).__name__, direct_load_error)
    print()
    print("Fallback: load base model through Unsloth and attach adapter")

    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    model = PeftModel.from_pretrained(
        base_model,
        str(ADAPTER_DIR),
        is_trainable=False,
    )

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

# Включает оптимизированный inference Unsloth.
FastLanguageModel.for_inference(model)
model.eval()

print("Model is ready for Unsloth inference.")
print("PAD token:", tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token_id)

Adapter directory: /kaggle/input/models/olllllllll/lora7b-150/pytorch/default/1
Base model: unsloth/qwen2.5-coder-7b-instruct-bnb-4bit


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded adapter directly through FastLanguageModel
Model is ready for Unsloth inference.
PAD token: 151665
EOS token: 151645


In [32]:
def clean_generated_sql(text: str) -> str:
    text = text.strip()

    if "```sql" in text.lower():
        start = text.lower().index("```sql") + len("```sql")
        text = text[start:].split("```", 1)[0]

    elif "```" in text:
        text = text.split("```", 1)[1].split("```", 1)[0]

    stop_markers = [
        "### Explanation:",
        "Explanation:",
        "\n\nExplanation",
        "\n###",
        "\nNote:",
        "\nThe query",
        "\nThis query",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker, 1)[0]

    text = " ".join(text.strip().split())

    if text.endswith(";"):
        text = text[:-1].strip()

    return text


def format_schema_for_repair(db_id: str) -> str:
    schema = SPIDER_SCHEMAS[db_id]

    lines = ["Valid schema:"]

    for table_name, table_info in sorted(schema["tables"].items()):
        columns = ", ".join(
            sorted(table_info["columns"])
        )

        lines.append(
            f"- {table_name}({columns})"
        )

    lines.append("")
    lines.append("Known foreign-key joins:")

    formatted_edges = []

    for edge in schema["foreign_keys"]:
        refs = sorted(edge)

        if len(refs) != 2:
            continue

        formatted_edges.append(
            f"- {refs[0][0]}.{refs[0][1]} "
            f"= {refs[1][0]}.{refs[1][1]}"
        )

    lines.extend(
        sorted(set(formatted_edges))
        if formatted_edges
        else ["- none listed"]
    )

    return "\n".join(lines)


def build_repair_prompt(
    item: dict,
    original_sql: str,
    report: GuardReport,
) -> str:
    return f"""Correct the SQLite query.

Question:
{item["question"]}

{format_schema_for_repair(item["db_id"])}

Original SQL:
{original_sql}

SchemaGuard report:
{json.dumps(asdict(report), ensure_ascii=False, indent=2)}

Rules:
- Return exactly one executable SQLite query.
- Use only tables and columns listed in Valid schema.
- Do not invent identifiers.
- Use the minimum number of tables needed.
- Add a JOIN only when necessary.
- Prefer Known foreign-key joins.
- Preserve the meaning of the question.
- Use SQLite syntax. Use LIMIT instead of TOP.
- Return SQL only.

### SQL:
"""


def generate_repaired_sql(prompt: str) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=ALLOW_REPAIR_PROMPT_TRUNCATION,
        max_length=(
            MAX_SEQ_LENGTH
            if ALLOW_REPAIR_PROMPT_TRUNCATION
            else None
        ),
    )

    prompt_tokens = int(
        inputs["input_ids"].shape[1]
    )

    if (
        prompt_tokens > MAX_SEQ_LENGTH
        and not ALLOW_REPAIR_PROMPT_TRUNCATION
    ):
        raise ValueError(
            f"Repair prompt слишком длинный: "
            f"{prompt_tokens} > {MAX_SEQ_LENGTH}. "
            "Увеличьте MAX_SEQ_LENGTH."
        )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:,
    ]

    raw_sql = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    return clean_generated_sql(raw_sql)


def is_repair_better(
    original_report: GuardReport,
    repaired_report: GuardReport,
) -> bool:
    if not repaired_report.sqlite_valid:
        return False

    if repaired_report.hard_error_count > 0:
        return False

    return repaired_report.score < original_report.score

In [33]:
RESULTS_JSONL = (
    PRED_DIR
    / "old_predictions_vs_guarded.jsonl"
)

results = []

for item, old_sql, original_report in tqdm(
    zip(eval_examples, old_predictions, audit_reports),
    total=len(eval_examples),
):
    old_sql = old_sql.strip()

    guarded_sql = old_sql
    repaired_sql = ""
    repaired_report = None
    repair_attempted = False
    repair_accepted = False
    repair_error = ""

    if should_attempt_repair(original_report):
        repair_attempted = True

        try:
            repaired_sql = generate_repaired_sql(
                build_repair_prompt(
                    item=item,
                    original_sql=old_sql,
                    report=original_report,
                )
            )

            repaired_report = guard_sql(
                item["db_id"],
                repaired_sql,
            )

            if is_repair_better(
                original_report,
                repaired_report,
            ):
                guarded_sql = repaired_sql
                repair_accepted = True

        except Exception as error:
            repair_error = (
                f"{type(error).__name__}: {error}"
            )

    result = {
        "id": item["id"],
        "db_id": item["db_id"],
        "question": item["question"],
        "gold_sql": item["sql"],
        "old_sql": old_sql,
        "old_guard": asdict(original_report),
        "repaired_sql": repaired_sql,
        "repaired_guard": (
            asdict(repaired_report)
            if repaired_report is not None
            else None
        ),
        "guarded_sql": guarded_sql,
        "repair_attempted": repair_attempted,
        "repair_accepted": repair_accepted,
        "repair_error": repair_error,
    }

    results.append(result)

with open(RESULTS_JSONL, "w", encoding="utf-8") as file:
    for row in results:
        file.write(
            json.dumps(row, ensure_ascii=False)
            + "\n"
        )

summary = {
    "examples": len(results),
    "repair_attempted": sum(
        row["repair_attempted"]
        for row in results
    ),
    "repair_accepted": sum(
        row["repair_accepted"]
        for row in results
    ),
    "changed_sql": sum(
        row["old_sql"] != row["guarded_sql"]
        for row in results
    ),
    "repair_errors": sum(
        bool(row["repair_error"])
        for row in results
    ),
}

print(json.dumps(
    summary,
    ensure_ascii=False,
    indent=2,
))

print("Saved:", RESULTS_JSONL)

  0%|          | 0/1034 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

{
  "examples": 1034,
  "repair_attempted": 52,
  "repair_accepted": 28,
  "changed_sql": 28,
  "repair_errors": 0
}
Saved: /kaggle/working/old_predictions_schema_guard/predictions/old_predictions_vs_guarded.jsonl


In [34]:
GOLD = EVAL_DIR / "gold.sql"
BASELINE = EVAL_DIR / "old_baseline_pred.sql"
GUARDED = EVAL_DIR / "guarded_pred.sql"


def safe_sql(sql: str) -> str:
    sql = " ".join(
        sql.strip()
        .replace("\n", " ")
        .split()
    )

    return sql if sql else "SELECT 1"


with open(GOLD, "w", encoding="utf-8") as gold_file, \
     open(BASELINE, "w", encoding="utf-8") as baseline_file, \
     open(GUARDED, "w", encoding="utf-8") as guarded_file:

    for row in results:
        gold_file.write(
            f"{safe_sql(row['gold_sql'])}"
            f"\t{row['db_id']}\n"
        )

        baseline_file.write(
            safe_sql(row["old_sql"])
            + "\n"
        )

        guarded_file.write(
            safe_sql(row["guarded_sql"])
            + "\n"
        )

print("Gold:", GOLD)
print("Baseline:", BASELINE)
print("Guarded:", GUARDED)

Gold: /kaggle/working/old_predictions_schema_guard/eval_files/gold.sql
Baseline: /kaggle/working/old_predictions_schema_guard/eval_files/old_baseline_pred.sql
Guarded: /kaggle/working/old_predictions_schema_guard/eval_files/guarded_pred.sql


In [35]:
SPIDER_REPO = Path(
    "/kaggle/working/spider_official"
)

TEST_SUITE_REPO = Path(
    "/kaggle/working/test_suite_eval"
)


def clone_if_missing(
    repository_url: str,
    destination: Path,
) -> None:
    if destination.exists():
        print("Already exists:", destination)
        return

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            repository_url,
            str(destination),
        ],
        check=True,
    )


clone_if_missing(
    "https://github.com/taoyds/spider.git",
    SPIDER_REPO,
)

clone_if_missing(
    "https://github.com/taoyds/test-suite-sql-eval.git",
    TEST_SUITE_REPO,
)

evaluation_file = (
    SPIDER_REPO
    / "evaluation.py"
)

source_lines = evaluation_file.read_text(
    encoding="utf-8"
).splitlines()

patched_lines = []

for index, line in enumerate(source_lines):
    patched_lines.append(line)

    if line.strip() != "conn = sqlite3.connect(db)":
        continue

    next_line = (
        source_lines[index + 1].strip()
        if index + 1 < len(source_lines)
        else ""
    )

    if "text_factory" in next_line:
        continue

    indentation = (
        line[:len(line) - len(line.lstrip())]
    )

    patched_lines.append(
        indentation
        + 'conn.text_factory = '
        + 'lambda raw: raw.decode(errors="ignore")'
    )

evaluation_file.write_text(
    "\n".join(patched_lines) + "\n",
    encoding="utf-8",
)

print("Spider evaluator UTF-8 patch: OK")

Already exists: /kaggle/working/spider_official
Already exists: /kaggle/working/test_suite_eval
Spider evaluator UTF-8 patch: OK


In [36]:
def run_evaluator(
    command: list,
    log_name: str,
) -> str:
    command = [
        str(item)
        for item in command
    ]

    result = subprocess.run(
        command,
        text=True,
        capture_output=True,
        check=False,
    )

    full_output = (
        f"RETURN CODE: {result.returncode}\n\n"
        f"[STDOUT]\n{result.stdout}\n\n"
        f"[STDERR]\n{result.stderr}\n"
    )

    log_path = (
        LOG_DIR
        / log_name
    )

    log_path.write_text(
        full_output,
        encoding="utf-8",
    )

    print("=" * 120)
    print(full_output)

    if result.returncode != 0:
        raise RuntimeError(
            "Evaluator завершился с ошибкой. "
            f"Смотрите лог: {log_path}"
        )

    return result.stdout


def evaluate_predictions(
    repository: Path,
    prediction_file: Path,
    log_name: str,
) -> str:
    return run_evaluator(
        [
            sys.executable,
            repository / "evaluation.py",
            "--gold", GOLD,
            "--pred", prediction_file,
            "--db", SPIDER_DB_DIR,
            "--table", SPIDER_TABLES,
            "--etype", "all",
        ],
        log_name,
    )


spider_baseline_output = evaluate_predictions(
    SPIDER_REPO,
    BASELINE,
    "spider_old_baseline.txt",
)

spider_guarded_output = evaluate_predictions(
    SPIDER_REPO,
    GUARDED,
    "spider_guarded.txt",
)

test_suite_baseline_output = evaluate_predictions(
    TEST_SUITE_REPO,
    BASELINE,
    "test_suite_old_baseline.txt",
)

test_suite_guarded_output = evaluate_predictions(
    TEST_SUITE_REPO,
    GUARDED,
    "test_suite_guarded.txt",
)

RETURN CODE: 0

[STDOUT]
eval_err_num:1
medium pred: SELECT T1.Name , T1.Song_release_year FROM singer AS T1 JOIN (SELECT MIN(Age) FROM singer) AS T2 ON T1.Age = T2.MIN
medium gold: SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:2
medium pred: SELECT T1.song_name , T1.song_release_year FROM singer AS T2 JOIN song AS T1 ON T2.singer_id = T1.singer_id WHERE T2.age = (SELECT MIN(age) FROM singer)
medium gold: SELECT song_name , song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT max(capacity) , avg(capacity) FROM stadium
medium gold: select max(capacity), average from stadium

medium pred: SELECT name , capacity FROM stadium WHERE average = ( SELECT MAX ( average ) FROM stadium )
medium gold: SELECT name , capacity FROM stadium ORDER BY average DESC LIMIT 1

medium pred: SELECT name , capacity FROM stadium WHERE average = ( SELECT MAX ( average ) FROM stadium )
medium gold: SELECT name , capacity FROM stadium ORDER BY average DESC L

In [37]:
def extract_metric(
    output: str,
    label: str,
):
    for line in output.splitlines():
        stripped = line.strip()

        if not stripped.lower().startswith(
            label.lower()
        ):
            continue

        values = re.findall(
            r"\d+\.\d+",
            stripped,
        )

        if values:
            # Последнее число соответствует колонке all.
            return float(values[-1])

    return None


comparison = {
    "schema_guard_mode": {
        "repair_warnings": REPAIR_WARNINGS,
    },
    "audit_before_repair": audit_summary,
    "repair": summary,
    "spider_official": {
        "baseline_execution_accuracy": extract_metric(
            spider_baseline_output,
            "execution",
        ),
        "guarded_execution_accuracy": extract_metric(
            spider_guarded_output,
            "execution",
        ),
        "baseline_exact_match": extract_metric(
            spider_baseline_output,
            "exact match",
        ),
        "guarded_exact_match": extract_metric(
            spider_guarded_output,
            "exact match",
        ),
    },
    "test_suite": {
        "baseline_execution_accuracy": extract_metric(
            test_suite_baseline_output,
            "execution",
        ),
        "guarded_execution_accuracy": extract_metric(
            test_suite_guarded_output,
            "execution",
        ),
        "baseline_exact_match": extract_metric(
            test_suite_baseline_output,
            "exact match",
        ),
        "guarded_exact_match": extract_metric(
            test_suite_guarded_output,
            "exact match",
        ),
    },
}

for section_name in [
    "spider_official",
    "test_suite",
]:
    section = comparison[section_name]

    section["execution_accuracy_delta"] = (
        section["guarded_execution_accuracy"]
        - section["baseline_execution_accuracy"]
    )

    section["exact_match_delta"] = (
        section["guarded_exact_match"]
        - section["baseline_exact_match"]
    )

comparison_path = (
    OUTPUT_DIR
    / "comparison.json"
)

comparison_path.write_text(
    json.dumps(
        comparison,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(json.dumps(
    comparison,
    ensure_ascii=False,
    indent=2,
))

print("Saved:", comparison_path)

{
  "schema_guard_mode": {
    "repair_warnings": false
  },
  "audit_before_repair": {
    "examples": 1034,
    "sqlite_invalid": 51,
    "unknown_tables": 10,
    "unknown_aliases": 10,
    "unknown_columns": 45,
    "ambiguous_columns": 59,
    "suspicious_joins": 50,
    "repair_candidates": 52
  },
  "repair": {
    "examples": 1034,
    "repair_attempted": 52,
    "repair_accepted": 28,
    "changed_sql": 28,
    "repair_errors": 0
  },
  "spider_official": {
    "baseline_execution_accuracy": 0.749,
    "guarded_execution_accuracy": 0.766,
    "baseline_exact_match": 0.699,
    "guarded_exact_match": 0.713,
    "execution_accuracy_delta": 0.017000000000000015,
    "exact_match_delta": 0.014000000000000012
  },
  "test_suite": {
    "baseline_execution_accuracy": 0.779,
    "guarded_execution_accuracy": 0.798,
    "baseline_exact_match": 0.699,
    "guarded_exact_match": 0.713,
    "execution_accuracy_delta": 0.019000000000000017,
    "exact_match_delta": 0.014000000000000012
  